# Immune PCA and cancer/treatment comparisons

This notebook fills missing immune measurements with missMDA, runs PCA, and tests associations with cancer type and treatment. It saves the results used by Figure 2, Extended Figure 2 and Figure 3.

We exclude samples missing eight or more of the 11 features, or whose observed feature values sum to zero.

- **Original:** 75 complete samples from 47 patients.
- **missMDA:** 235 samples from 84 patients, including all 75 Original samples.

Only immune measurements are used for imputation and PCA. Patient, age, cancer, treatment and outcome labels are not used to fill missing values or calculate the PCA axes.

The PCA uses all timepoints. Predicting future outcomes would need separate validation by patient.


In [ ]:
# Run from the repository root or from notebooks/.
candidates <- c(getwd(), file.path(getwd(), ".."))
is_project <- vapply(candidates, function(path) {
  file.exists(file.path(path, "data", "metadata.rds")) &&
    dir.exists(file.path(path, "R"))
}, logical(1))
if (!any(is_project)) stop("Open this notebook from the repository root or notebooks/.")
project_root <- normalizePath(candidates[which(is_project)[1]])

required <- c("missMDA", "lme4", "lmerTest", "emmeans")
missing <- required[!vapply(required, requireNamespace, quietly = TRUE,
                           FUN.VALUE = logical(1))]
if (length(missing)) stop("Install analysis packages listed in R/README.md: ",
                          paste(missing, collapse = ", "))
source(file.path(project_root, "R/functions", "pca_analysis.R"))
source(file.path(project_root, "R/functions", "pca_associations.R"))
source(file.path(project_root, "R/functions", "pca_figure_io.R"))
tables_dir <- file.path(project_root, "tables", "pca_analysis")
dir.create(tables_dir, recursive = TRUE, showWarnings = FALSE)


## Missing values and PCA

The 11 features are ATC, Naïve%, SCM%, CM%, EM%, TE%, PD1%, LAG3%, TIM3%, PBMC TCR Shannon diversity and cfDNA TCR Shannon diversity. PBMC and cfDNA samples are matched by `sample_id`.

missMDA fills missing entries using two components (`ncp = 2`). Observed values stay unchanged. The features are centred and scaled before PCA.

Both datasets use the PCA axes fitted to the 235 completed samples. PC1 points towards higher TE%, and PC2 towards higher Naïve%. This only sets the direction of the axes.

The optional check below hides 10% of the measurements 20 times and compares ranks 0–8. It tests how well missing values can be reconstructed. The main analysis keeps rank 2.


In [ ]:
inputs <- prepare_pca_inputs(project_root, ncp = 2L, seed = 20261012L)
preparation_audit <- write_pca_source_data(inputs, tables_dir)
print(preparation_audit, row.names = FALSE)
stopifnot(nrow(inputs$raw235) == 235L,
          nrow(inputs$real_features75) == 75L,
          sum(is.na(inputs$raw235)) == 247L,
          max(abs(inputs$completed235[!is.na(inputs$raw235)] -
                    inputs$raw235[!is.na(inputs$raw235)])) == 0)
print(table(inputs$scoredata310$dataset, inputs$scoredata310$cycle))


In [ ]:
# Optional: computationally heavier reconstruction-rank sensitivity.
# The fixed-rank inputs above remain the source for the manuscript figures.
RUN_RANK_CV <- FALSE
if (RUN_RANK_CV) {
  cv_inputs <- prepare_pca_inputs(project_root, rerun_rank_cv = TRUE)
  print(cv_inputs$rank_cv)
  write_pca_source_data(cv_inputs, file.path(tables_dir, "rank_crossvalidation"))
}


## Cancer and treatment comparisons

**Panel d:** Compare baseline PC1 between lymphoma and non-lymphoma, using one X01 sample per patient. The model adjusts for age and uses robust standard errors (HC3). ALCL is included in lymphoma.

**Panels e–f:** Compare baseline and treatment PC scores using a patient random intercept to account for repeated samples. An additional model adjusts for age and cancer group. Each Xk sample is assigned the treatment from cycle k−1.

P values use Holm correction:

- Baseline comparisons: 2 tests across the two datasets.
- Treatment comparisons: 30 tests across both PCs and datasets, separately for the main and age/cancer-adjusted models.
- Paired checks: 4 tests.

Model confidence intervals use Bonferroni adjustment across the same families: 2 baseline comparisons and 30 treatment comparisons per analysis. These are 95% simultaneous intervals. P values still use Holm correction. The original pointwise limits are kept in the tables. Paired-check bootstrap intervals describe the median and rank effect; they are pointwise and do not correspond to the Wilcoxon test.

The paired check compares X01 with the same patient's first eligible treatment sample. The raincloud plots show all relevant samples.

These comparisons cannot establish that a drug caused a change. The confidence intervals also do not include uncertainty from filling missing values.


In [ ]:
statistics <- fit_pca_associations(inputs)
write_pca_association_tables(statistics, tables_dir)
print(statistics$families, row.names = FALSE)
print(statistics$baseline_lymphoma[, c("dataset", "n", "n_lymphoma", "estimate",
                                      "lower", "upper", "p_Holm")], row.names = FALSE)
selected <- statistics$regimen_contrasts
selected <- selected[selected$analysis == "Current_patient_intercept_primary" &
  ((selected$regimen == "Vincristine/Cyclophosphamide" & selected$PC == "PC1") |
     (selected$regimen == "Vincristine" & selected$PC == "PC2")), ]
print(selected[, c("dataset", "PC", "regimen", "estimate", "lower.CL", "upper.CL",
                   "p.value", "p_Holm")], row.names = FALSE)
print(statistics$diagnostics, row.names = FALSE)
# Singular Original-PC2 fits are retained and explicitly flagged, as in the
# existing analysis; use the sensitivity tables when assessing robustness.


## Save results

- `data/pca_inputs.rds`: immune measurements, sample details, PCA model, scores and loadings.
- `data/association_results.rds`: effect estimates, confidence intervals, adjusted P values and additional checks.

Figure 2 uses both files. Extended Figure 2 and Figure 3 use `pca_inputs.rds`. Additional analysis tables are saved in `tables/pca_analysis/`.


In [ ]:
write_pca_figure_outputs(inputs, statistics, project_root)
prepared <- read_pca_figure_outputs(project_root)
stopifnot(isTRUE(all.equal(prepared$inputs$completed235, inputs$completed235)),
          isTRUE(all.equal(prepared$inputs$reference_pca$x, inputs$reference_pca$x)),
          isTRUE(all.equal(prepared$statistics$baseline_lymphoma,
                           statistics$baseline_lymphoma)),
          isTRUE(all.equal(prepared$statistics$regimen_contrasts,
                           statistics$regimen_contrasts)))
cat("Prepared PCA inputs and statistical results saved in data/.\n")
